# E036 — Multi-Resolution PCA for Face Recognition

**Motivation:** Standard PCA operates on full 80×80 images (6400d → 50d). Multi-resolution approach may capture both coarse facial structure and fine details better.

**Hypothesis:** Pyramid PCA (extract PCA features at multiple resolutions and concatenate) will improve robustness to session variation.

**Approach:**
1. Create image pyramid: 80×80, 40×40, 20×20
2. Extract PCA features at each level
3. Concatenate multi-scale features
4. Train LogReg on concatenated features

**Configs:**
- `single`: E007 baseline (80×80 → 50d)
- `pyramid_2`: 80×80 + 40×40 (50d + 25d = 75d)
- `pyramid_3`: 80×80 + 40×40 + 20×20 (50d + 25d + 12d = 87d)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E007_REF = {'mean_eer': 0.97, 'std_eer': 0.86}

In [ ]:
def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def resize_image(x, size):
    """Resize 80×80 image to size×size."""
    img = Image.fromarray(x.astype(np.uint8))
    img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32)

def extract_pyramid_features(x, levels=[80, 40, 20], n_pca=[50, 25, 12]):
    """Extract multi-resolution PCA features."""
    features = []
    for size, n_comp in zip(levels, n_pca):
        if size == 80:
            x_resized = x
        else:
            x_resized = resize_image(x, size)
        features.append(x_resized.flatten())
    return np.concatenate(features)

def _aug_flip(x): return x[:, ::-1].copy()
def _aug_bright(x, rng): return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)
def _aug_inoise(x, rng): return np.clip(x + rng.normal(0, 15, x.shape), 0, 255)

def train_pyramid_model(df, data_dir, pyramid_config, seed):
    """Train multi-resolution PCA model."""
    rng = np.random.default_rng(seed)
    levels, n_pca = pyramid_config
    
    X, y = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        vecs = [orig]
        vecs += [_aug_flip(orig), _aug_bright(orig, rng), _aug_inoise(orig, rng)]
        
        for v in vecs:
            feat = extract_pyramid_features(v, levels, n_pca)
            X.append(feat); y.append(row["label"])
    
    X = np.array(X)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(X_scaled, y)
    
    return scaler, clf

def score_pyramid(png_path, scaler, clf, pyramid_config):
    x = _load_image(png_path)
    feat = extract_pyramid_features(x, pyramid_config[0], pyramid_config[1])
    feat_scaled = scaler.transform(feat.reshape(1, -1))
    return float(clf.decision_function(feat_scaled)[0])

def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

print('Pyramid functions defined')

## 2. Cross-validation

In [ ]:
configs = {
    'single': ([80], [6400]),  # No PCA, full features
    'pyramid_2': ([80, 40], [50, 25]),
    'pyramid_3': ([80, 40, 20], [50, 25, 12]),
}

results = {}

for name, config in configs.items():
    print(f"\n=== {name} ===")
    fold_eers = []
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        
        scaler, clf = train_pyramid_model(train_df, DATA, config, seed_f)
        
        scores = []
        for _, row in val_df.iterrows():
            score = score_pyramid(_find_png(row["stem"], DATA), scaler, clf, config)
            scores.append(score)
        
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    results[name] = {
        'fold_eers': fold_eers,
        'mean': np.mean(fold_eers),
        'std': np.std(fold_eers),
    }
    print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for name, res in results.items():
    print(f"{name:12s}: {res['mean']:5.2f} ± {res['std']:5.2f}%")

## 3. Conclusion

Multi-resolution PCA: [TBD]
Decision: [Adopt/Reject]